# Step 0 — Data Acquisition, Cleaning & NER

This notebook **retroactively documents** how the Music KB seed data was acquired.

Pipeline:
1. Crawl artist/album/label data from the **MusicBrainz API** (no key needed)
2. Clean and normalise the raw JSON
3. Run **spaCy NER** to extract and validate named entities
4. Export the canonical seed list used in Step 1
5. Document 3 real ambiguity cases

In [ ]:
!pip install requests pandas spacy tqdm --quiet
!python -m spacy download en_core_web_sm --quiet

## 1. MusicBrainz API Crawler

MusicBrainz is an open music encyclopedia with a free REST API (`musicbrainz.org/ws/2`).
Rate limit: 1 request/second — we respect this with `time.sleep(1)`.

In [7]:
import time, requests, json, re
from tqdm.notebook import tqdm
import pandas as pd

MB_BASE = "https://musicbrainz.org/ws/2"
HEADERS = {"User-Agent": "MusicKB-LabProject/1.0 (lab@university.edu)"}

def mb_get(endpoint, params={}):
    """Single MusicBrainz API call with polite rate-limiting."""
    params["fmt"] = "json"
    r = requests.get(f"{MB_BASE}/{endpoint}", params=params,
                     headers=HEADERS, timeout=15)
    r.raise_for_status()
    time.sleep(1.1)   # respect 1 req/s limit
    return r.json()

# ── Seed artists to crawl (same 20) ────────────────────────────
SEED_ARTISTS = [
    "Beyoncé", "Michael Jackson", "David Bowie", "Aretha Franklin",
    "Eric Clapton", "Jay-Z", "Kendrick Lamar", "Adele", "Drake",
    "Taylor Swift", "Radiohead", "Nina Simone", "Bob Marley",
    "Daft Punk", "Burna Boy", "Prince", "Billie Eilish",
    "Frank Ocean", "Paul McCartney", "Miles Davis"
]

def search_artist(name):
    """Search MusicBrainz for an artist by name, return best match."""   
    data = mb_get("artist", {"query": f'artist:"{name}"', "limit": 3})
    artists = data.get("artists", [])
    if not artists:
        return None
    # Pick highest score
    best = max(artists, key=lambda a: int(a.get("score", 0)))
    return best

print("Searching for seed artists...")
raw_artists = {}
for name in tqdm(SEED_ARTISTS):
    result = search_artist(name)
    if result:
        raw_artists[name] = result
        print(f"  ✅  {name:25} → MBID: {result['id']}  score: {result.get('score')}")
    else:
        print(f"  ❌  {name} — not found")

print(f"\nFound {len(raw_artists)}/{len(SEED_ARTISTS)} artists.")


Searching for seed artists...


  0%|          | 0/20 [00:00<?, ?it/s]

  ✅  Beyoncé                   → MBID: 859d0860-d480-4efd-970c-c05d5f1776b8  score: 100
  ✅  Michael Jackson           → MBID: f27ec8db-af05-4f36-916e-3d57f91ecf5e  score: 100
  ✅  David Bowie               → MBID: 5441c29d-3602-4898-b1a1-b77fa23b8e50  score: 100
  ✅  Aretha Franklin           → MBID: 2f9ecbed-27be-40e6-abca-6de49d50299e  score: 100
  ✅  Eric Clapton              → MBID: 618b6900-0618-4f1e-b835-bccb17f84294  score: 100
  ✅  Jay-Z                     → MBID: f82bcf78-5b69-4622-a5ef-73800768d9ac  score: 100
  ✅  Kendrick Lamar            → MBID: 381086ea-f511-4aba-bdf9-71c753dc5077  score: 100
  ✅  Adele                     → MBID: cc2c9c3c-b7bc-4b8b-84d8-4fbd8779e493  score: 100
  ✅  Drake                     → MBID: 9fff2f8a-21e6-47de-a2b8-7f449929d43f  score: 100
  ✅  Taylor Swift              → MBID: 20244d07-534f-4eff-b4d4-930878889970  score: 100
  ✅  Radiohead                 → MBID: a74b1b7f-71a5-4011-9441-d0b5e4122711  score: 100
  ✅  Nina Simone               →

## 2. Fetch artist details + discography

In [8]:
def get_artist_details(mbid):
    """Fetch full artist record including relations and tags."""    
    return mb_get(f"artist/{mbid}",
                  {"inc": "url-rels+tags+aliases+ratings"})

def get_artist_albums(mbid, limit=10):
    """Fetch top release-groups (albums) for an artist."""    
    data = mb_get("release-group",
                  {"artist": mbid, "type": "album", "limit": limit})
    return data.get("release-groups", [])

artist_details = {}
artist_albums  = {}

print("Fetching details & albums...")
for name, artist in tqdm(raw_artists.items()):
    mbid = artist["id"]
    try:
        details = get_artist_details(mbid)
        albums  = get_artist_albums(mbid, limit=8)
        artist_details[mbid] = details
        artist_albums[mbid]  = albums
    except Exception as e:
        print(f"  [WARN] {name}: {e}")

print(f"\nDetails fetched for {len(artist_details)} artists.")
print(f"Total albums found: {sum(len(v) for v in artist_albums.values())}")


Fetching details & albums...


  0%|          | 0/20 [00:00<?, ?it/s]


Details fetched for 20 artists.
Total albums found: 160


## 3. Cleaning pipeline

Problems found in raw API data:
- Inconsistent country codes (`XW`, `XE` = world/europe)
- Null birth/death years
- Duplicate aliases
- Tags with very low vote counts (noisy genre labels)

In [13]:
COUNTRY_NORM = {
    "US": "USA", "GB": "UK", "CA": "Canada",
    "AU": "Australia", "JM": "Jamaica", "FR": "France",
    "NG": "Nigeria", "XW": "International", "XE": "Europe",
    None: "Unknown"
}

MIN_TAG_VOTES = 2   # ignore tags with fewer votes

def clean_artist(raw):
    """Normalise a raw MusicBrainz artist dict."""    
    area = raw.get("area", {}) or {}
    country_code = raw.get("country") or area.get("iso-3166-1-codes", [None])[0]
    country = COUNTRY_NORM.get(country_code, country_code or "Unknown")

    life = raw.get("life-span", {}) or {}
    birth_year = None
    if life.get("begin"):
        try:
            birth_year = int(str(life["begin"])[:4])
        except:
            pass

    tags = [t["name"] for t in raw.get("tags", [])
            if int(t.get("count", 0)) >= MIN_TAG_VOTES]

    return {
        "mbid":       raw["id"],
        "name":       raw.get("name", ""),
        "sort_name":  raw.get("sort-name", ""),
        "type":       raw.get("type", "Person"),
        "country":    country,
        "birth_year": birth_year,
        "tags":       tags[:5],   # keep top 5 tags as genre hints
        "disambiguation": raw.get("disambiguation", ""),
    }

def clean_album(raw, artist_mbid):
    """Normalise a raw release-group dict."""    
    year = None
    date = raw.get("first-release-date", "")
    if date:
        try:
            year = int(str(date)[:4])
        except:
            pass
    return {
        "mbid":        raw["id"],
        "title":       raw.get("title", ""),
        "type":        raw.get("primary-type", "Album"),
        "year":        year,
        "artist_mbid": artist_mbid,
    }

cleaned_artists = {}
cleaned_albums  = []

for mbid, raw in artist_details.items():
    cleaned_artists[mbid] = clean_artist(raw)

for mbid, albums in artist_albums.items():
    for alb in albums:
        cleaned_albums.append(clean_album(alb, mbid))

df_artists = pd.DataFrame(cleaned_artists.values())
df_albums  = pd.DataFrame(cleaned_albums).drop_duplicates("mbid")

print("Cleaned artists:")
display(df_artists[["name","type","country","birth_year","tags"]].head(20))
print(f"\nCleaned albums: {len(df_albums)}")
display(df_albums.head(10))


Cleaned artists:


,name,type,country,birth_year,tags
0,Beyoncé,Person,USA,1981,"[2000s, 2010s, 2020s, adult contemporary, amer..."
1,Michael Jackson,Person,USA,1958,"[contemporary r&b, dance-pop, disco, new jack ..."
2,David Bowie,Person,UK,1947,"[actors, alternative rock, art pop, art rock, ..."
3,Aretha Franklin,Person,USA,1942,"[deep soul, gospel, soul, southern soul]"
4,Eric Clapton,Person,UK,1945,"[blues, blues rock, rock, soft rock]"
5,JAY‐Z,Person,USA,1969,"[boom bap, east coast hip hop, hip hop, pop rap]"
6,Kendrick Lamar,Person,USA,1987,"[2010s, 2020s, alternative hip hop, conscious ..."
7,Adele,Person,UK,1988,"[blue-eyed soul, neo soul, pop, pop soul, soul]"
8,Drake,Person,Canada,1986,"[alternative r&b, contemporary r&b, hip hop, p..."
9,Taylor Swift,Person,USA,1989,"[2000s, 2010s, 2020s, alternative pop, artist ..."



Cleaned albums: 160


,mbid,title,type,year,artist_mbid
0,2c385052-5083-43a2-b1e5-36566d2ae3c0,RENAISSANCE,Album,2022,859d0860-d480-4efd-970c-c05d5f1776b8
1,40eaa07b-7542-48e7-887c-b5e686964a7a,4,Album,2011,859d0860-d480-4efd-970c-c05d5f1776b8
2,6f8e8119-de1b-39f0-85ca-b5252cb9211d,I Am… Sasha Fierce,Album,2008,859d0860-d480-4efd-970c-c05d5f1776b8
3,8596171e-23b2-4bdd-8f1c-11960df0ecbe,BEYONCÉ,Album,2013,859d0860-d480-4efd-970c-c05d5f1776b8
4,9fdbccb2-9b51-36b7-b643-c7ae586b797d,B’Day,Album,2006,859d0860-d480-4efd-970c-c05d5f1776b8
5,c1f22e07-7bdf-4a4f-8b50-7747c1091ef6,Lemonade,Album,2016,859d0860-d480-4efd-970c-c05d5f1776b8
6,f639ba46-b371-3262-b494-5f2dbee36f5c,Dangerously in Love,Album,2003,859d0860-d480-4efd-970c-c05d5f1776b8
7,fa566e7b-254a-4fab-8aa5-ebcb7592d7f2,COWBOY CARTER,Album,2024,859d0860-d480-4efd-970c-c05d5f1776b8
8,06b064b9-01e7-32d8-b585-86404584e795,Music & Me,Album,1973,f27ec8db-af05-4f36-916e-3d57f91ecf5e
9,50b9d7de-9124-33c0-83a3-76722bf346e5,"Forever, Michael",Album,1975,f27ec8db-af05-4f36-916e-3d57f91ecf5e


## 4. NER with spaCy

We run spaCy `en_core_web_sm` over artist bios and album titles to:
- Validate entity types (PERSON, ORG, GPE, WORK_OF_ART)
- Surface aliases and alternate names
- Flag ambiguous entities

In [14]:
import spacy
nlp = spacy.load("en_core_web_sm")

# Build short bio sentences from cleaned data to feed NER
def make_bio(artist):
    name   = artist["name"]
    ctype  = artist["type"]
    ctry   = artist["country"]
    year   = artist["birth_year"] or "unknown year"
    tags   = ", ".join(artist["tags"]) if artist["tags"] else "various genres"
    disamb = f' ({artist["disambiguation"]})' if artist["disambiguation"] else ""
    return (f"{name}{disamb} is a {ctype.lower()} from {ctry}, "
            f"born in {year}, known for {tags}.")

bios = [make_bio(a) for a in cleaned_artists.values()]

print("NER results on artist bios:\n")
ner_rows = []
for bio in bios[:20]:
    doc = nlp(bio)
    ents = [(e.text, e.label_) for e in doc.ents]
    print(f"  TEXT: {bio[:80]}...")
    print(f"  ENTS: {ents}\n")
    for text, label in ents:
        ner_rows.append({"bio_snippet": bio[:60], "entity": text, "label": label})

df_ner = pd.DataFrame(ner_rows)
print("\nNER label distribution:")
print(df_ner["label"].value_counts().to_string())


NER results on artist bios:

  TEXT: Beyoncé is a person from USA, born in 1981, known for 2000s, 2010s, 2020s, adult...
  ENTS: [('Beyoncé', 'ORG'), ('USA', 'GPE'), ('1981', 'DATE'), ('2000s', 'DATE'), ('2010s,', 'DATE'), ('american', 'NORP')]

  TEXT: Michael Jackson (“King of Pop”) is a person from USA, born in 1958, known for co...
  ENTS: [('Michael Jackson', 'PERSON'), ('King of Pop', 'WORK_OF_ART'), ('USA', 'GPE'), ('1958', 'DATE'), ('r&b', 'PERSON'), ('jack swing', 'PERSON')]

  TEXT: David Bowie (English singer‐songwriter) is a person from UK, born in 1947, known...
  ENTS: [('David Bowie', 'PERSON'), ('English', 'LANGUAGE'), ('UK', 'GPE'), ('1947', 'DATE'), ('british', 'NORP')]

  TEXT: Aretha Franklin is a person from USA, born in 1942, known for deep soul, gospel,...
  ENTS: [('Aretha Franklin', 'PERSON'), ('USA', 'GPE'), ('1942', 'DATE')]

  TEXT: Eric Clapton is a person from UK, born in 1945, known for blues, blues rock, roc...
  ENTS: [('Eric Clapton', 'PERSON'), ('UK',

## 5. Three Ambiguity Cases

Required by the grading guide — real examples from the MusicBrainz data.

In [15]:
ambiguity_cases = [
    {
        "case": "PERSON vs ORG — 'Prince'",
        "problem": (
            "Searching MusicBrainz for 'Prince' returns both the solo artist "
            "(Prince Rogers Nelson, Q7542) and 'Prince' as a band name and "
            "as part of album titles. spaCy tags 'Prince' as PERSON in biographical "
            "context but as ORG when it appears as a label name."
        ),
        "resolution": (
            "We disambiguate using MusicBrainz entity type field ('Person' vs 'Group') "
            "and cross-check with the Wikidata QID obtained in Step 2. "
            "Confidence threshold 0.99 ensures only the solo artist is linked."
        ),
    },
    {
        "case": "Album vs Song — 'Thriller'",
        "problem": (
            "MusicBrainz returns both the album 'Thriller' (Q174305) and the "
            "single 'Thriller' as separate release-groups. spaCy labels both as "
            "WORK_OF_ART with identical surface form, giving no signal to distinguish them."
        ),
        "resolution": (
            "We filter by release-group primary-type == 'Album' and require "
            "first-release-date to match our known year (1982). "
            "The entity type MUS:Album is enforced in the ontology."
        ),
    },
    {
        "case": "Country code ambiguity — 'XW' / 'XE'",
        "problem": (
            "MusicBrainz uses ISO 3166 extension codes: 'XW' (Worldwide) and "
            "'XE' (Europe) for acts with no single national origin (e.g. Daft Punk "
            "is listed under both FR and XW depending on the release). "
            "spaCy NER does not recognise these codes as GPE entities at all."
        ),
        "resolution": (
            "We apply a manual normalisation map (COUNTRY_NORM dict above) "
            "and default to the artist's birth country (France for Daft Punk) "
            "when the release-level code is XW/XE."
        ),
    },
]

for i, case in enumerate(ambiguity_cases, 1):
    print(f"{'='*60}")
    print(f"Case {i}: {case['case']}")
    print(f"\nProblem:\n  {case['problem']}")
    print(f"\nResolution:\n  {case['resolution']}\n")


Case 1: PERSON vs ORG — 'Prince'

Problem:
  Searching MusicBrainz for 'Prince' returns both the solo artist (Prince Rogers Nelson, Q7542) and 'Prince' as a band name and as part of album titles. spaCy tags 'Prince' as PERSON in biographical context but as ORG when it appears as a label name.

Resolution:
  We disambiguate using MusicBrainz entity type field ('Person' vs 'Group') and cross-check with the Wikidata QID obtained in Step 2. Confidence threshold 0.99 ensures only the solo artist is linked.

Case 2: Album vs Song — 'Thriller'

Problem:
  MusicBrainz returns both the album 'Thriller' (Q174305) and the single 'Thriller' as separate release-groups. spaCy labels both as WORK_OF_ART with identical surface form, giving no signal to distinguish them.

Resolution:
  We filter by release-group primary-type == 'Album' and require first-release-date to match our known year (1982). The entity type MUS:Album is enforced in the ontology.

Case 3: Country code ambiguity — 'XW' / 'XE'

Prob

## 6. Export seed data

In [12]:
df_artists.to_csv("seed_artists.csv", index=False)
df_albums.to_csv("seed_albums.csv", index=False)
df_ner.to_csv("ner_entities.csv", index=False)

print("Saved:")
print("  seed_artists.csv    —", len(df_artists), "artists")
print("  seed_albums.csv     —", len(df_albums),  "albums")
print("  ner_entities.csv    —", len(df_ner),      "NER entities")
print("\nCrawler ethics summary:")
print("  - Rate limited to 1 req/s (MusicBrainz requirement)")
print("  - User-Agent header identifies the project")
print("  - Only public CC0-licensed data accessed")
print("  - No login / scraping of protected pages")


Saved:
  seed_artists.csv    — 20 artists
  seed_albums.csv     — 160 albums
  ner_entities.csv    — 86 NER entities

Crawler ethics summary:
  - Rate limited to 1 req/s (MusicBrainz requirement)
  - User-Agent header identifies the project
  - Only public CC0-licensed data accessed
  - No login / scraping of protected pages


In [1]:
!jupyter nbconvert --to html step0_crawler_ner.ipynb

[NbConvertApp] Converting notebook step0_crawler_ner.ipynb to html
[NbConvertApp] Writing 336260 bytes to step0_crawler_ner.html
